In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
df_bt = pd.read_csv("files/welsh_back_translation_high_similarity.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
                .rename(columns={"back_translated_welsh": "text"})

In [11]:
df_bt

,text,cefr_level
0,Roedd Ma/ penbwrdd: fy nhad yn awr yn mynd i'r...,A1
1,Ar y acw: Byddwch yn troi at y dde yma. Ewch o...,A1
2,Ac: mae prynhawn da. Ar y tywydd yn ofnadwy! Y...,A1
3,Ble ydych chi'n byw?,A1
4,Beth ydych chi'n hoffi?,A1
...,...,...
365,Dylen nhw aros yn aros.,A2
366,Hoffwn i fynd i Affrica.,A2
367,A hoffech chi fynd i America?,A2
368,A fydden nhw'n mynd i'r Almaen?,A2


In [12]:
# Add missing columns 
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [13]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
0,Roedd Ma/ penbwrdd: fy nhad yn awr yn mynd i'r...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,Ar y acw: Byddwch yn troi at y dde yma. Ewch o...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Ac: mae prynhawn da. Ar y tywydd yn ofnadwy! Y...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,Ble ydych chi'n byw?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,Beth ydych chi'n hoffi?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
365,Dylen nhw aros yn aros.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
366,Hoffwn i fynd i Affrica.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
367,A hoffech chi fynd i America?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
368,A fydden nhw'n mynd i'r Almaen?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [14]:
# Count how many samples are labeled A1 and A2
df_bt["cefr_level"].value_counts()

cefr_level
A1    243
A2    127
Name: count, dtype: int64

In [15]:
# Load Welsh CEFR dataset from HuggingFace
ds_welsh = load_dataset("UniversalCEFR/learn_welsh_cy")["train"].to_pandas()  

In [16]:
ds_welsh

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,..."
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ..."
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b..."
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D..."
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...
...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?


In [17]:
df_merged = pd.concat([ds_welsh, df_bt], ignore_index=True)
df_merged = df_merged.drop_duplicates(subset="text", keep="first").sample(frac=1, random_state=42)

In [18]:
df_merged

,title,lang,source_name,format,category,cefr_level,license,text
1399,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A1,CC-BY-SA,Edrychais ar y noson olaf.
745,Uned 27 - Pris y petrol,cy,mynediad-de-learnwelsh,sentence-level,reference,A1,public,Brawd Siân dw i.
1652,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A2,CC-BY-SA,Mae'n well gyda mi oren.
49,Uned 17 - Sgwrs,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Derbynnydd: Bore da, gaf i helpu?\nChris: Gaf ..."
189,Uned 4 - Ble dych chi'n mynd? 11,cy,mynediad-de-learnwelsh,sentence-level,reference,A1,public,Dyn ni'n mynd i'r gwely.
...,...,...,...,...,...,...,...,...
1133,Uned 10 - Wnei di?,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Wnei di helpu gyda'r gwaith?
1297,Uned 17 -na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Caerdydd yw'r ddinas fwya swnllyd.
863,Uned 17 - Cwrs Sbaeneg,cy,sylfaen-de-learnwelsh,document-level,reference,A2,public,Croeso i bawb... os dych chi'n siarad tipyn o'...
1487,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A1,CC-BY-SA,Oedd hi'n araf?


In [24]:
df_merged["cefr_level"].value_counts()

cefr_level
A1    964
A2    719
Name: count, dtype: int64

In [19]:
# Save the filtered DataFrame to a CSV file
df_merged.to_csv("files/checkDA_welsh.csv", index=False)

In [20]:
hf_dataset=Dataset.from_pandas(df_merged.reset_index(drop=True))

In [21]:
hf_dataset

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 1683
})

In [25]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in hf_dataset["cefr_level"]])

In [26]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [27]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [28]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [29]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [30]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(hf_dataset, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = hf_dataset.select(train_idx)
    ds_val = hf_dataset.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_DA/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 337/337 [00:00<00:00, 8087.89 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_40096\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.826800,0.733805,0.611276,0.549141,0.622521,0.611276,0.606164,0.917098,0.729897,0.644444,0.201389,0.306878,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.563400,0.494663,0.741840,0.734714,0.744853,0.741840,0.732456,0.865285,0.793349,0.761468,0.576389,0.656126,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.280100,0.401464,0.854599,0.851831,0.860906,0.854599,0.827273,0.943005,0.881356,0.905983,0.736111,0.812261,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 2...


Map: 100%|██████████| 337/337 [00:00<00:00, 15957.29 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_40096\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Map: 100%|██████████| 336/336 [00:00<00:00, 11557.87 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_40096\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.811700,0.660568,0.610119,0.556337,0.611806,0.610119,0.609155,0.896373,0.725367,0.615385,0.223776,0.328205,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.418100,0.334429,0.866071,0.866343,0.867002,0.866071,0.893617,0.870466,0.881890,0.831081,0.860140,0.845361,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.159100,0.323761,0.886905,0.886559,0.886793,0.886905,0.889447,0.917098,0.903061,0.883212,0.846154,0.864286,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 336/336 [00:00<00:00, 15762.24 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_40096\3235310919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,0.785700,0.446619,0.797619,0.798402,0.801221,0.797619,0.844444,0.791667,0.817204,0.743590,0.805556,0.773333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.339300,0.304950,0.877976,0.877549,0.877904,0.877976,0.879397,0.911458,0.895141,0.875912,0.833333,0.854093,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.132700,0.299393,0.910714,0.910241,0.911440,0.910714,0.900990,0.947917,0.923858,0.925373,0.861111,0.892086,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [31]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_BTwelsh_DA/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [32]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)


In [33]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.860906  0.854599  0.851831  0.827273  0.943005  0.881356   
1        2        0.774916  0.774481  0.774671  0.806283  0.797927  0.802083   
2        3        0.869365  0.869436  0.868897  0.870647  0.906736  0.888325   
3        4        0.886793  0.886905  0.886559  0.889447  0.917098  0.903061   
4        5        0.911440  0.910714  0.910241  0.900990  0.947917  0.923858   
5  Average        0.860684  0.859227  0.858440  0.858928  0.902537  0.879737   

         A2                      ...   B1        B2                    C1  \
  Precision    Recall        F1  ...   F1 Precision Recall   F1 Precision   
0  0.905983  0.736111  0.812261  ...  0.0       0.0    0.0  0.0       0.0   
1  0.732877  0.743056  0.737931  ...  0.0       0.0    0.0  0.0       0.0   
2  0.867647  0.819444  0.842857  ...  0.0       0.0    0.0  0.0       0.0   
3  0.883212  0.846154  0.864286  ...  0.0       0.0    0.0  0.0       0.0   
4  0.925373  0.861111  0.892086  ...  0.0       0.0    0.0  0.0       0.0   
5  0.863018  0.801175  0.829884  ...  0.0       0.0    0.0  0.0       0.0   

                     C2              
  Recall   F1 Precision Recall   F1  
0    0.0  0.0       0.0    0.0  0.0  
1    0.0  0.0       0.0    0.0  0.0  
2    0.0  0.0       0.0    0.0  0.0  
3    0.0  0.0       0.0    0.0  0.0  
4    0.0  0.0       0.0    0.0  0.0  
5    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]